# 04 — Building a Multimodal Benchmark Harness

> **Orbit 5 (Multimodal)** · **Difficulty**: Intermediate · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/04_building_a_multimodal_benchmark_harness.ipynb)

**Prerequisites**: [01–03 multimodal](01_intro_to_multimodal_systems.ipynb), [evals/01](../evals/01_evals_why_measurement_matters.ipynb)

**What you will learn**:
- Why multimodal evals are harder than text-only evals
- How to design a benchmark dataset with ground truth across modalities
- Building a reusable harness: loader → inference → parser → scorer → reporter
- Evaluation metrics for visual grounding, extraction, and reasoning

In [ ]:
# --- Setup ---
!pip install -q "transformers>=4.57.0" accelerate bitsandbytes torch numpy pandas matplotlib Pillow

import json, math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter, defaultdict

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)
print("Dependencies ready.")

## 1 — Why Multimodal Evals Are Harder

Text evals compare strings. Multimodal evals must assess:

| Dimension | Text eval | Multimodal eval |
|-----------|----------|-----------------|
| **Correctness** | String match / LLM judge | String match + spatial accuracy + temporal alignment |
| **Grounding** | Citation check | Bounding box IoU, timestamp overlap |
| **Completeness** | All facts covered? | All visual elements identified? All audio segments captured? |
| **Hallucination** | Text not in source | Objects not in image, sounds not in audio |

The extra challenge: **ground truth is expensive**. Annotating bounding boxes, timestamps, and spatial relationships requires domain expertise.

In [ ]:
# Create a synthetic benchmark dataset
# In practice, you would curate real images and annotations
benchmark = [
    {
        "id": "img_qa_01",
        "task": "image_qa",
        "question": "What color is the largest shape?",
        "ground_truth": "red",
        "difficulty": "easy",
    },
    {
        "id": "img_qa_02",
        "task": "image_qa",
        "question": "How many shapes are in the image?",
        "ground_truth": "3",
        "difficulty": "medium",
    },
    {
        "id": "chart_01",
        "task": "chart_extraction",
        "question": "What is the value for Category A?",
        "ground_truth": "45",
        "difficulty": "easy",
    },
    {
        "id": "chart_02",
        "task": "chart_extraction",
        "question": "Which category has the highest value?",
        "ground_truth": "Category C",
        "difficulty": "medium",
    },
    {
        "id": "doc_01",
        "task": "document_extraction",
        "question": "What is the invoice total?",
        "ground_truth": "$5,234.00",
        "difficulty": "easy",
    },
    {
        "id": "spatial_01",
        "task": "spatial_reasoning",
        "question": "Is the blue shape inside the red shape?",
        "ground_truth": "yes",
        "difficulty": "medium",
    },
    {
        "id": "count_01",
        "task": "counting",
        "question": "How many red objects are there?",
        "ground_truth": "2",
        "difficulty": "hard",
    },
    {
        "id": "compare_01",
        "task": "comparison",
        "question": "Which shape is larger, the blue or the green?",
        "ground_truth": "blue",
        "difficulty": "medium",
    },
]

df_bench = pd.DataFrame(benchmark)
print(f"Benchmark: {len(benchmark)} examples")
print(f"Tasks: {df_bench['task'].value_counts().to_dict()}")
print(f"Difficulty: {df_bench['difficulty'].value_counts().to_dict()}")
print()
print(df_bench[['id', 'task', 'question', 'ground_truth']].to_string(index=False))

## 2 — The Harness Architecture

A good benchmark harness separates concerns into composable stages:

```
Dataset (examples with ground truth)
     │
     ▼
┌───────────────┐
│  Input Loader  │  <- Load/prepare inputs per example
└──────┬────────┘
       │
       ▼
┌───────────────┐
│  Model Runner  │  <- Run inference, collect raw output
└──────┬────────┘
       │
       ▼
┌───────────────┐
│ Output Parser  │  <- Extract structured answer from raw text
└──────┬────────┘
       │
       ▼
┌───────────────┐
│    Scorer      │  <- Compare parsed output vs ground truth
└──────┬────────┘
       │
       ▼
┌───────────────┐
│   Reporter    │  <- Aggregate scores, visualize, export
└───────────────┘
```

In [ ]:
# Build each harness component

class BenchmarkHarness:
    def __init__(self, dataset):
        self.dataset = dataset
        self.results = []

    def load_input(self, example):
        return {
            "question": example["question"],
            "task": example["task"],
        }

    def run_model(self, inputs):
        # Placeholder: in practice, call your VLM here
        # For demo, return simulated outputs
        simulated = {
            "img_qa_01": "red",
            "img_qa_02": "3",
            "chart_01": "45",
            "chart_02": "Category C",
            "doc_01": "$5,234.00",
            "spatial_01": "yes",
            "count_01": "3",  # Wrong answer to demonstrate scoring
            "compare_01": "blue",
        }
        return simulated.get(inputs.get("id", ""), "unknown")

    def parse_output(self, raw_output):
        return raw_output.strip().lower()

    def score(self, predicted, ground_truth):
        pred = predicted.strip().lower()
        gt = ground_truth.strip().lower()
        # Exact match
        exact = pred == gt
        # Contains match (more lenient)
        contains = gt in pred or pred in gt
        return {"exact_match": exact, "contains_match": contains}

    def run(self):
        self.results = []
        for example in self.dataset:
            inputs = self.load_input(example)
            inputs["id"] = example["id"]
            raw = self.run_model(inputs)
            parsed = self.parse_output(raw)
            scores = self.score(parsed, example["ground_truth"])
            self.results.append({
                "id": example["id"],
                "task": example["task"],
                "difficulty": example["difficulty"],
                "ground_truth": example["ground_truth"],
                "predicted": parsed,
                **scores,
            })
        return self.results

    def report(self):
        df = pd.DataFrame(self.results)
        print("=== Overall Results ===")
        print(f"Exact match accuracy: {df['exact_match'].mean():.1%}")
        print(f"Contains match accuracy: {df['contains_match'].mean():.1%}")
        print()
        print("=== By Task ===")
        print(df.groupby("task")["exact_match"].mean().to_string())
        print()
        print("=== By Difficulty ===")
        print(df.groupby("difficulty")["exact_match"].mean().to_string())
        return df

harness = BenchmarkHarness(benchmark)
harness.run()
results_df = harness.report()

## 3 — Evaluation Metrics for Multimodal

Beyond exact match, multimodal tasks need specialized metrics:

| Metric | Formula | Use case |
|--------|---------|----------|
| **Exact Match** | pred == ground_truth | Factoid QA, classification |
| **F1 Score** | Token-level precision/recall | Extraction tasks |
| **IoU** | Intersection / Union of bounding boxes | Grounding, localization |
| **WER** | Edit distance / reference length | Transcription quality |
| **BLEU/ROUGE** | N-gram overlap | Summarization, description |

In [ ]:
# Implement key metrics from scratch

def exact_match(pred, gt):
    return pred.strip().lower() == gt.strip().lower()

def token_f1(pred, gt):
    pred_tokens = set(pred.lower().split())
    gt_tokens = set(gt.lower().split())
    if not gt_tokens:
        return 1.0 if not pred_tokens else 0.0
    if not pred_tokens:
        return 0.0
    common = pred_tokens & gt_tokens
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gt_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def bbox_iou(box_a, box_b):
    # boxes: [x1, y1, x2, y2]
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

# Test the metrics
print("Exact match:")
print(f"  'red' vs 'red': {exact_match('red', 'red')}")
print(f"  'Red' vs 'red': {exact_match('Red', 'red')}")
print(f"  'dark red' vs 'red': {exact_match('dark red', 'red')}")

print("\nToken F1:")
print(f"  'the cat sat' vs 'the cat sat': {token_f1('the cat sat', 'the cat sat'):.2f}")
print(f"  'a cat sat' vs 'the cat sat on mat': {token_f1('a cat sat', 'the cat sat on mat'):.2f}")

print("\nBBox IoU:")
print(f"  [0,0,100,100] vs [0,0,100,100]: {bbox_iou([0,0,100,100], [0,0,100,100]):.2f}")
print(f"  [0,0,100,100] vs [50,50,150,150]: {bbox_iou([0,0,100,100], [50,50,150,150]):.2f}")
print(f"  [0,0,100,100] vs [200,200,300,300]: {bbox_iou([0,0,100,100], [200,200,300,300]):.2f}")

In [ ]:
# Visualize benchmark results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# By task
task_acc = results_df.groupby("task")["exact_match"].mean().sort_values()
axes[0].barh(task_acc.index, task_acc.values, color="#4CAF50")
axes[0].set_xlabel("Exact Match Accuracy")
axes[0].set_title("Accuracy by Task Type")
axes[0].set_xlim(0, 1.1)

# By difficulty
diff_acc = results_df.groupby("difficulty")["exact_match"].mean().reindex(["easy", "medium", "hard"])
axes[1].bar(diff_acc.index, diff_acc.values, color=["#4CAF50", "#FF9800", "#F44336"])
axes[1].set_ylabel("Exact Match Accuracy")
axes[1].set_title("Accuracy by Difficulty")
axes[1].set_ylim(0, 1.1)

# IoU demonstration
boxes = [
    ([10, 10, 60, 60], [10, 10, 60, 60], "Perfect overlap"),
    ([10, 10, 60, 60], [30, 30, 80, 80], "Partial overlap"),
    ([10, 10, 60, 60], [70, 70, 120, 120], "No overlap"),
]
ious = [bbox_iou(a, b) for a, b, _ in boxes]
labels = [l for _, _, l in boxes]
axes[2].bar(labels, ious, color=["#4CAF50", "#FF9800", "#F44336"])
axes[2].set_ylabel("IoU Score")
axes[2].set_title("Bounding Box IoU Examples")
axes[2].set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

## 4 — Extending the Harness for Real Models

To use this harness with a real VLM, replace the `run_model` method:

```python
class VLMHarness(BenchmarkHarness):
    def __init__(self, dataset, model, processor):
        super().__init__(dataset)
        self.model = model
        self.processor = processor

    def run_model(self, inputs):
        messages = [{"role": "user", "content": [
            {"type": "image", "image": inputs["image"]},
            {"type": "text", "text": inputs["question"]},
        ]}]
        # Process and generate
        text = self.processor.apply_chat_template(messages, ...)
        inputs_t = self.processor(text=[text], images=[inputs["image"]], ...)
        output = self.model.generate(**inputs_t, max_new_tokens=100)
        return self.processor.decode(output[0], skip_special_tokens=True)
```

The harness architecture stays the same — only the model interface changes.

## Exercises

1. **Extend the dataset**: Add 10 examples covering a new task type (e.g., table extraction, spatial counting). Include ground truth.

2. **Add a metric**: Implement ROUGE-L (longest common subsequence) and add it to the scorer.

3. **Compare models**: Run the same benchmark with 2 different prompt templates. How much does prompt design affect accuracy?

In [ ]:
# Exercise starter code

# Exercise 1: Add new examples
new_examples = [
    {"id": "table_01", "task": "table_extraction",
     "question": "YOUR QUESTION", "ground_truth": "ANSWER", "difficulty": "medium"},
]

# Exercise 2: ROUGE-L implementation
def rouge_l(pred, gt):
    # Implement longest common subsequence
    pred_tokens = pred.lower().split()
    gt_tokens = gt.lower().split()
    # LCS dynamic programming
    m, n = len(pred_tokens), len(gt_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred_tokens[i-1] == gt_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs_len = dp[m][n]
    precision = lcs_len / m if m > 0 else 0
    recall = lcs_len / n if n > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return f1

print(f"ROUGE-L test: {rouge_l('the cat sat on the mat', 'the cat sat on a mat'):.3f}")

## Key Takeaways

| # | Insight |
|---|---------|
| 1 | Multimodal evals must assess **correctness + grounding + completeness** |
| 2 | Build harnesses with **composable stages**: load → infer → parse → score → report |
| 3 | Use **task-appropriate metrics**: exact match, F1, IoU, WER depending on the task |
| 4 | Ground truth annotation is **the bottleneck** — invest in quality labels |
| 5 | **Measure before optimizing** — you can’t improve what you don’t measure |

## What’s Next

→ [05_image_prompting_and_visual_reasoning.ipynb](05_image_prompting_and_visual_reasoning.ipynb) — Visual prompting patterns

## References

- Yue et al., “MMBench: Is Your Multi-modal Model an All-around Player?” (2023)
- Liu et al., “MMStar: Are We on the Right Way for Evaluating Large Vision-Language Models?” (2024)
- Goyal et al., “Making the V in VQA Matter” (2017)